# Study 2: the hybrid gate (OWLv2 + Grounding DINO) on fresh scenes

Code of **Sections V-A, V-B and V-E** (Tables IV and V). Only the detector changes; weights, thresholds and the decision flow are
those of Study 1. The hybrid routes the bowls, the ramekin and the plate to OWLv2 and the cabinet, the stove and the cookie box to
Grounding DINO.

* **Development** = initial states 0-2 (210 scenes), **fresh test** = initial states 5-9 (350 scenes: 150 impossible, 200 valid),
  untouched by any earlier experiment.
* The detector settings were frozen in `frozen_config_v6.json` (hash `a5f5591cec`, 2 Oct 2026) before any fresh scene was run.
* The cells are in the order we ran them. Cells marked *before the freeze* refuse to run once the frozen file exists (on purpose).

*The printed output below each cell is the output of our own run (paths shortened). Long progress logs are shortened to their first and last lines.*

In [ ]:
# Shared set-up (about 3 min): result folders, models, simulator and the gate's code. Run it once per session.
%run -i setup/s0_paths.py
%run -i setup/s1_models_and_simulator.py
%run -i setup/s2_engine.py
%run -i setup/s3_calibrate_camera.py
%run -i setup/s4_engine_fixes.py
%run -i setup/s5_policy_helpers.py
%run -i setup/s6_study1_setup.py
%run -i setup/s7_study2_setup.py

## Detector inside the gate, step 1: OWLv2 alone (70 scenes of initial state 0, resumable)
Table IV, first rows. Also tests OWLv2 with the contrast check.

In [ ]:
# %% CELL v6_owl_gate — same frozen gate, OWLv2 instead of Grounding DINO. No robot rollouts. Resumable.
# Runs 2 versions on the chosen start states and compares them with the old Grounding DINO gate on the SAME scenes:
#   owl          : OWLv2 detector, everything else as before
#   owl_contrast : OWLv2 + the contrast check switched back on (a box is not "stove" if it looks more like "cabinet")
import copy
from transformers import Owlv2Processor, Owlv2ForObjectDetection
V6_DIR = os.path.join(RESULTS_ROOT, "v6")
os.makedirs(V6_DIR, exist_ok=True)
OWL_STATES = [0]                     # [0] = 70 scenes (quick test).  Use [0, 1, 2] for 210.
OWL_RAMP = (0.10, 0.25)              # OWLv2 scores are lower than Grounding DINO's: 0.10 = "seen", 0.25 = "sure"
VARIANTS = {"owl": {"scene_vocab": []},
            "owl_contrast": {"scene_vocab": ["black bowl", "plate", "ramekin", "cookie box", "wooden cabinet", "stove"]}}

if "_owl_mdl" not in globals():
    _owl_proc = Owlv2Processor.from_pretrained("google/owlv2-base-patch16-ensemble")
    _owl_mdl = Owlv2ForObjectDetection.from_pretrained("google/owlv2-base-patch16-ensemble").to(device).eval()
    print("OWLv2 loaded")

def ground_object_owl(image_pil, phrase, box_threshold=0.10, text_threshold=None):
    """drop-in replacement for ground_object(): same output format, OWLv2 inside"""
    inp = _owl_proc(text=[["a photo of a " + phrase.lower().strip()]], images=image_pil, return_tensors="pt",
                    truncation=True, max_length=16).to(device)
    with torch.no_grad():
        out = _owl_mdl(**inp)
    ts = torch.tensor([[max(image_pil.size), max(image_pil.size)]], device=device)
    try:
        r = _owl_proc.post_process_grounded_object_detection(outputs=out, target_sizes=ts, threshold=box_threshold)[0]
    except Exception:
        r = _owl_proc.post_process_object_detection(outputs=out, target_sizes=ts, threshold=box_threshold)[0]
    return [{"bbox": [round(v, 1) for v in b.tolist()], "confidence": round(float(s), 3), "label": phrase}
            for b, s in zip(r["boxes"], r["scores"])]

def owl_case(env, task_id, state, kind, clean_instr):
    instr = perturb_instruction(clean_instr, kind, state) if kind in ("absent_reference", "absent_target") else clean_instr
    obs = setup_episode(env, task_id, state, perturb=kind if kind in SCENE_PERTURBATIONS else None, instruction=instr)
    tkey, gkey = _true_keys(env, obs)
    parsed, goal = parse_instruction(instr), parse_goal(instr)
    cam = get_camera_model(env, obs)
    refs = _as_list(parsed.get("reference"))
    rel = parsed.get("relation")
    row = {"task_id": task_id, "state": state, "split": _split(state), "kind": kind,
           "valid": int(kind not in INVALID_KINDS_V4), "instruction": instr, "relation": rel,
           "has_goal": int(bool(goal and goal.get("object"))),
           "label_uncertain": int((kind == "position_change" and any("plate" in (r or "") for r in refs))
                                  or (kind == "distractor" and rel == "near")),
           "t_policy_s": 0.0, "seconds_total": 0.0}
    for sfx, att in (("0", 0), ("1", 1)):
        t0 = time.time()
        sc = perceive_scene(env, obs, parsed, goal, cam, att, CFG)
        row[f"t_perceive{sfx}_s"] = round(time.time() - t0, 3)
        row.update(_scene_fields(sc, 1.0, sfx))                      # the action term has weight 0 in the frozen gate
        sel, rt, rr, rg, _ = _grade_scene(sc, obs, cam, parsed, kind, tkey, gkey)
        row.update({f"sel_{sfx}": sel, f"right_target_{sfx}": rt, f"right_ref_{sfx}": rr, f"right_goal_{sfx}": rg})
    return row

if "_GO_REAL" not in globals():
    _GO_REAL = ground_object                                         # the real Grounding DINO function (kept once)
_CFG_SAVE = {k: copy.deepcopy(CFG[k]) for k in ("min_conf", "ground_min_conf", "conf_floor", "conf_full", "scene_vocab")}
try:
    ground_object = ground_object_owl
    CFG.update({"min_conf": [OWL_RAMP[0]] * 3, "ground_min_conf": OWL_RAMP[0], "conf_floor": OWL_RAMP[0], "conf_full": OWL_RAMP[1]})
    for vname, vcfg in VARIANTS.items():
        CFG.update(vcfg)
        path = os.path.join(V6_DIR, f"verify_{vname}.csv")
        done = _done_keys(path, ["task_id", "state", "kind"])
        n_todo = sum((str(t), str(s), k) not in done for t in range(NUM_TASKS) for s in OWL_STATES for k in KINDS)
        print(f"\n=== {vname}: {n_todo} scenes to run ({len(done)} already saved)")
        i, t_start = 0, time.time()
        for task_id in range(NUM_TASKS):
            todo = [(s, k) for s in OWL_STATES for k in KINDS if (str(task_id), str(s), k) not in done]
            if not todo:
                continue
            env, _, clean_instr = get_task_env_and_instruction(task_id)
            for s, k in todo:
                row = owl_case(env, task_id, s, k, clean_instr)
                row["frozen_hash"] = vname
                _append(path, VERIFY_FIELDS, row)
                i += 1
                print(f"[{i}/{n_todo}] t{task_id} s{s} {k:18s} G={row['G_0']:.2f} S_rel={row['S_rel_0']:.2f} D_rel={row['D_rel_0']:.2f} "
                      f"sel={row['sel_0']:10s} (~{(time.time() - t_start) / i * (n_todo - i) / 60:.1f} min left)")
            env.close()
finally:
    ground_object = _GO_REAL                                         # always put Grounding DINO and the old settings back
    CFG.update(_CFG_SAVE)
    OBS_HOOK = None

# ---------------- comparison on the SAME scenes ----------------
_FROZEN = json.load(open(FROZEN_JSON))
_FCFG = {**_FROZEN, "weights": tuple(_FROZEN["weights"]), "ramp": tuple(_FROZEN["ramp"])}
_bsm = base_success_map()
def _load(path):
    d = pd.read_csv(path).drop_duplicates(["task_id", "state", "kind"], keep="last")
    return d[d.state.isin(OWL_STATES)].reset_index(drop=True)
RES = {"Grounding DINO (old)": decide_all(_load(VERIFY_CSV), _FCFG)}
for vname in VARIANTS:
    RES[vname] = decide_all(_load(os.path.join(V6_DIR, f"verify_{vname}.csv")), {**_FCFG, "ramp": OWL_RAMP})

rows = []
for name, dec in RES.items():
    s = summarize(dec, _bsm)
    val = dec[dec.kind.isin(VALID_KINDS)]
    rows.append({"gate": name, "scenes": len(dec),
                 "right bowl % (valid)": pct(pd.to_numeric(val.right_target_0, errors="coerce").mean()),
                 "right reference %": pct(pd.to_numeric(dec.right_ref_0, errors="coerce").mean()),
                 "RIGHT instructions accepted %": s["valid_accept %"],
                 "WRONG instructions acted on % (lower=better)": s["IER %"],
                 "nominal success %": s.get("clean_TSR %"), "AUROC": s["AUROC(C)"], "J": s["J"],
                 "seconds per scene": round(float((dec.t_perceive0_s + dec.t_perceive1_s * dec.reground_used).mean()), 2)})
summary = pd.DataFrame(rows)
print("\n\nGATE COMPARISON on start states", OWL_STATES, "(SmolVLA alone: accepts 100% / acts on 100%)")
print(summary.set_index("gate").T.to_string())
print("\nrobot allowed to act, per condition (%):")
print(pd.DataFrame({n: d.groupby("kind").executed.mean().mul(100).round(0) for n, d in RES.items()}).reindex(KINDS).to_string())
print("\nright bowl on valid scenes, per task (%):")
print(pd.DataFrame({n: pd.to_numeric(d[d.kind.isin(VALID_KINDS)].right_target_0, errors="coerce").groupby(d.task_id).mean().mul(100).round(0)
                    for n, d in RES.items()}).to_string())
summary.to_csv(os.path.join(V6_DIR, "gate_comparison_owl.csv"), index=False)
_old = summary.iloc[0]
for _, r in summary.iloc[1:].iterrows():
    ok = (r["RIGHT instructions accepted %"] > _old["RIGHT instructions accepted %"]
          and r["WRONG instructions acted on % (lower=better)"] <= _old["WRONG instructions acted on % (lower=better)"])
    print(f"\n{r['gate']}: {'POSITIVE (accepts more right instructions, acts on no more wrong ones)' if ok else 'NOT clearly positive'}")

OWLv2 loaded

=== owl: 0 scenes to run (70 already saved)

=== owl_contrast: 0 scenes to run (70 already saved)


GATE COMPARISON on start states [0] (SmolVLA alone: accepts 100% / acts on 100%)
gate                                          Grounding DINO (old)     owl  owl_contrast
scenes                                                     70.0000  70.000        70.000
right bowl % (valid)                                       32.5000  62.500        62.500
right reference %                                          50.0000  58.300        66.700
RIGHT instructions accepted %                              42.5000  32.500        32.500
WRONG instructions acted on % (lower=better)               13.3000   0.000         0.000
nominal success %                                          50.0000  40.000        40.000
AUROC                                                       0.6730   0.752         0.742
J                                                           0.2917   0.325         0.325
seco

## Detector inside the gate, step 2: more words per object, and the hybrid (70 scenes, resumable)
Table IV: OWLv2 with extra words and the hybrid detector.

In [ ]:
# %% CELL v6_owl_gate_fix — two fixes for the objects OWLv2 misses (cabinet, stove, cookie box, moved plate). Resumable.
#   owl_words  : OWLv2 is asked with several words per object ("wooden cabinet" / "cabinet" / "drawer", "plate" / "dish", ...)
#                and may keep bigger boxes (the cabinet covers more than 25% of the picture)
#   owl_hybrid : OWLv2 for bowls / ramekin / plate; Grounding DINO for cabinet, stove, cookie box
#                (and for the plate only when OWLv2 finds none)
assert "ground_object_owl" in globals() and "owl_case" in globals(), "Run the v6_owl_gate cell first (it loads OWLv2)."

OWL_WORDS = {"wooden cabinet": ["wooden cabinet", "cabinet", "drawer"],
             "stove": ["stove", "stove burner", "hot plate"],
             "cookie box": ["cookie box", "box", "snack box"],
             "plate": ["plate", "dish", "white plate"]}

def ground_object_owl_words(image_pil, phrase, box_threshold=0.10, text_threshold=None):
    words = OWL_WORDS.get(phrase.lower().strip(), [phrase.lower().strip()])
    inp = _owl_proc(text=[["a photo of a " + w for w in words]], images=image_pil, return_tensors="pt",
                    truncation=True, max_length=16).to(device)
    with torch.no_grad():
        out = _owl_mdl(**inp)
    ts = torch.tensor([[max(image_pil.size), max(image_pil.size)]], device=device)
    try:
        r = _owl_proc.post_process_grounded_object_detection(outputs=out, target_sizes=ts, threshold=box_threshold)[0]
    except Exception:
        r = _owl_proc.post_process_object_detection(outputs=out, target_sizes=ts, threshold=box_threshold)[0]
    return [{"bbox": [round(v, 1) for v in b.tolist()], "confidence": round(float(s), 3), "label": phrase}
            for b, s in zip(r["boxes"], r["scores"])]

GD_PHRASES = {"wooden cabinet", "cabinet", "drawer", "stove", "cooktop", "burner", "cookie box", "box", "carton"}
GD_FALLBACK = {"plate", "white plate", "dish"}

def _gd_rescaled(image_pil, phrase, box_threshold):
    """Grounding DINO boxes with their confidence moved onto OWLv2's scale (0.22-0.40 -> OWL_RAMP)."""
    out = []
    for d in _GO_REAL(image_pil, phrase, box_threshold=0.22, text_threshold=0.20):
        c = OWL_RAMP[0] + (float(d["confidence"]) - 0.22) * (OWL_RAMP[1] - OWL_RAMP[0]) / (0.40 - 0.22)
        if c >= box_threshold:
            out.append({**d, "confidence": round(float(c), 3)})
    return out

def ground_object_hybrid(image_pil, phrase, box_threshold=0.10, text_threshold=None):
    p = phrase.lower().strip()
    if p in GD_PHRASES:
        return _gd_rescaled(image_pil, phrase, box_threshold)
    dets = ground_object_owl(image_pil, phrase, box_threshold)
    if not dets and p in GD_FALLBACK:
        dets = _gd_rescaled(image_pil, phrase, box_threshold)
    return dets

VARIANTS2 = {"owl_words": (ground_object_owl_words, {"scene_vocab": [], "max_area_frac": 0.35}),
             "owl_hybrid": (ground_object_hybrid, {"scene_vocab": []})}

_CFG_SAVE2 = {k: copy.deepcopy(CFG[k]) for k in ("min_conf", "ground_min_conf", "conf_floor", "conf_full", "scene_vocab", "max_area_frac")}
try:
    for vname, (gfn, vcfg) in VARIANTS2.items():
        ground_object = gfn
        CFG.update(_CFG_SAVE2)
        CFG.update({"min_conf": [OWL_RAMP[0]] * 3, "ground_min_conf": OWL_RAMP[0], "conf_floor": OWL_RAMP[0], "conf_full": OWL_RAMP[1]})
        CFG.update(vcfg)
        path = os.path.join(V6_DIR, f"verify_{vname}.csv")
        done = _done_keys(path, ["task_id", "state", "kind"])
        n_todo = sum((str(t), str(s), k) not in done for t in range(NUM_TASKS) for s in OWL_STATES for k in KINDS)
        print(f"\n=== {vname}: {n_todo} scenes to run ({len(done)} already saved)")
        i, t_start = 0, time.time()
        for task_id in range(NUM_TASKS):
            todo = [(s, k) for s in OWL_STATES for k in KINDS if (str(task_id), str(s), k) not in done]
            if not todo:
                continue
            env, _, clean_instr = get_task_env_and_instruction(task_id)
            for s, k in todo:
                row = owl_case(env, task_id, s, k, clean_instr)
                row["frozen_hash"] = vname
                _append(path, VERIFY_FIELDS, row)
                i += 1
                print(f"[{i}/{n_todo}] t{task_id} s{s} {k:18s} G={row['G_0']:.2f} S_rel={row['S_rel_0']:.2f} D_rel={row['D_rel_0']:.2f} "
                      f"sel={row['sel_0']:10s} (~{(time.time() - t_start) / i * (n_todo - i) / 60:.1f} min left)")
            env.close()
finally:
    ground_object = _GO_REAL                                         # always put Grounding DINO and the old settings back
    CFG.update(_CFG_SAVE2)
    OBS_HOOK = None

# ---------------- comparison of ALL versions on the SAME scenes ----------------
RES2 = {"GDINO (old)": decide_all(_load(VERIFY_CSV), _FCFG)}
for vname in ["owl", "owl_contrast", "owl_words", "owl_hybrid"]:
    p = os.path.join(V6_DIR, f"verify_{vname}.csv")
    if os.path.exists(p):
        RES2[vname] = decide_all(_load(p), {**_FCFG, "ramp": OWL_RAMP})
rows = []
for name, dec in RES2.items():
    s = summarize(dec, _bsm)
    val = dec[dec.kind.isin(VALID_KINDS)]
    rows.append({"gate": name, "scenes": len(dec),
                 "right bowl % (valid)": pct(pd.to_numeric(val.right_target_0, errors="coerce").mean()),
                 "right reference %": pct(pd.to_numeric(dec.right_ref_0, errors="coerce").mean()),
                 "RIGHT instructions accepted %": s["valid_accept %"],
                 "WRONG instructions acted on % (lower=better)": s["IER %"],
                 "nominal success %": s.get("clean_TSR %"), "AUROC": s["AUROC(C)"], "J": s["J"]})
summary2 = pd.DataFrame(rows)
print("\n\nGATE COMPARISON on start states", OWL_STATES)
print(summary2.set_index("gate").T.to_string())
print("\nrobot allowed to act, per condition (%):")
print(pd.DataFrame({n: d.groupby("kind").executed.mean().mul(100).round(0) for n, d in RES2.items()}).reindex(KINDS).to_string())
print("\nRIGHT instructions accepted, per task (%):")
print(pd.DataFrame({n: d[d.valid == 1].groupby("task_id").executed.mean().mul(100).round(0) for n, d in RES2.items()}).to_string())
print("\nright bowl on valid scenes, per task (%):")
print(pd.DataFrame({n: pd.to_numeric(d[d.kind.isin(VALID_KINDS)].right_target_0, errors="coerce").groupby(d.task_id).mean().mul(100).round(0)
                    for n, d in RES2.items()}).to_string())
summary2.to_csv(os.path.join(V6_DIR, "gate_comparison_owl_fix.csv"), index=False)
_o = summary2.iloc[0]
for _, r in summary2.iloc[1:].iterrows():
    ok = (r["RIGHT instructions accepted %"] > _o["RIGHT instructions accepted %"]
          and r["WRONG instructions acted on % (lower=better)"] <= _o["WRONG instructions acted on % (lower=better)"])
    print(f"{r['gate']:14s}: {'POSITIVE' if ok else 'not clearly positive'}")


=== owl_words: 70 scenes to run (0 already saved)
[1/70] t0 s0 clean              G=1.00 S_rel=1.00 D_rel=1.00 sel=target     (~9.3 min left)
[2/70] t0 s0 absent_reference   G=0.00 S_rel=0.00 D_rel=0.00 sel=other_bowl (~8.9 min left)
... (191 more lines of progress) ...
owl           : not clearly positive
owl_contrast  : not clearly positive
owl_words     : not clearly positive
owl_hybrid    : POSITIVE


## Step 1: hybrid gate on all 210 development scenes (rule fixed in advance)
GO only if the hybrid accepts at least as many valid instructions and acts on no more impossible ones than the first gate.

In [ ]:
# %% CELL v6_B_confirm — STEP 1: hybrid gate on the 210 development scenes (~20 min, resumable, no robot)
run_gate_sweep(DEV6_STATES, "hybrid", HYB_DEV_CSV)
_dev = {"GDINO (old)": decide_all(load6(VERIFY_CSV, DEV6_STATES), gate_cfg6("gdino")),
        "hybrid (new)": decide_all(load6(HYB_DEV_CSV, DEV6_STATES), gate_cfg6("hybrid"))}
DEV_SUMMARY6 = compare_gates6(_dev, base_success_map(), f"STEP 1 - GATE COMPARISON on development states {DEV6_STATES}")
DEV_SUMMARY6.to_csv(os.path.join(V6_DIR, "step1_dev_comparison.csv"), index=False)
_a, _b, _p = paired6(_dev["GDINO (old)"], _dev["hybrid (new)"], 0)
print(f"\nwrong instructions: old acts where new stops in {_a} scenes, new acts where old stops in {_b} (exact McNemar p = {_p:.3f})")
_a, _b, _p = paired6(_dev["GDINO (old)"], _dev["hybrid (new)"], 1)
print(f"right instructions: old accepts where new refuses in {_a} scenes, new accepts where old refuses in {_b} (exact McNemar p = {_p:.3f})")
_o, _n = DEV_SUMMARY6.iloc[0], DEV_SUMMARY6.iloc[1]
STEP1_OK = bool(len(_dev["hybrid (new)"]) == len(DEV6_STATES) * NUM_TASKS * len(KINDS)
                and _n["RIGHT instructions accepted %"] >= _o["RIGHT instructions accepted %"]
                and _n["WRONG instructions acted on % (lower=better)"] <= _o["WRONG instructions acted on % (lower=better)"])
print("\nSTEP 1 VERDICT:", "GO  (hybrid accepts at least as many right instructions and acts on no more wrong ones) -> run the freeze cell"
      if STEP1_OK else "STOP - check this output before going on")


=== hybrid: 0 scenes to run (210 already saved) -> verify_owl_hybrid.csv

STEP 1 - GATE COMPARISON on development states [0, 1, 2]
gate                                          GDINO (old) hybrid (new)
scenes                                                210          210
right bowl % (valid)                                 31.7         70.0
right reference %                                    50.9         51.9
RIGHT instructions accepted %                        40.8         57.5
  ...accepted on the RIGHT bowl %                    21.7         45.0
WRONG instructions acted on % (lower=better)         10.0          6.7
  95% CI                                      (5.4, 17.9)  (3.1, 13.8)
nominal success %                                    40.0         50.0
AUROC                                               0.754        0.874
J                                                  0.3083       0.5083
seconds per scene                                    3.26         2.66

robot allowed t

## Step 1b: contrast check between cabinet, stove and cookie box (*before the freeze*)
Kept only if it acts on fewer-or-equal impossible instructions and accepts at least as many valid ones on the right bowl. It was dropped.

In [ ]:
# %% CELL v6_B2_contrast — STEP 1b: does the contrast check fix the "cabinet mistaken for cookie box / stove" bug?
# Same hybrid gate + one rule: a box is NOT accepted as "cookie box" / "stove" / "wooden cabinet" if Grounding DINO is
# more confident that the same box is one of the other two. ~25 min on the same 210 development scenes. Resumable.
# RULE FIXED BEFORE LOOKING: keep the contrast version only if it acts on FEWER-or-equal wrong instructions AND
# accepts at least as many right instructions ON THE RIGHT BOWL as the plain hybrid.
HYB_CON_CSV = os.path.join(V6_DIR, "verify_owl_hybrid_contrast.csv")
CONTRAST_VOCAB = ["cookie box", "wooden cabinet", "stove"]
assert not os.path.exists(FROZEN6_JSON), "The settings are already frozen - this test must come before the freeze."
_keep = list(HYB_SCENE_VOCAB)
try:
    HYB_SCENE_VOCAB = list(CONTRAST_VOCAB)
    run_gate_sweep(DEV6_STATES, "hybrid", HYB_CON_CSV)
finally:
    HYB_SCENE_VOCAB = _keep
_dev2 = {"GDINO (old)": decide_all(load6(VERIFY_CSV, DEV6_STATES), gate_cfg6("gdino")),
         "hybrid": decide_all(load6(HYB_DEV_CSV, DEV6_STATES), gate_cfg6("hybrid")),
         "hybrid + contrast": decide_all(load6(HYB_CON_CSV, DEV6_STATES), gate_cfg6("hybrid"))}
DEV_SUMMARY6B = compare_gates6(_dev2, base_success_map(), f"STEP 1b - CONTRAST CHECK on development states {DEV6_STATES}")
DEV_SUMMARY6B.to_csv(os.path.join(V6_DIR, "step1b_dev_contrast.csv"), index=False)
_a, _b, _p = paired6(_dev2["hybrid"], _dev2["hybrid + contrast"], 0)
print(f"\nwrong instructions: plain acts where contrast stops in {_a} scenes, contrast acts where plain stops in {_b} (exact McNemar p = {_p:.3f})")
_a, _b, _p = paired6(_dev2["hybrid"], _dev2["hybrid + contrast"], 1)
print(f"right instructions: plain accepts where contrast refuses in {_a} scenes, contrast accepts where plain refuses in {_b} (exact McNemar p = {_p:.3f})")
_h, _c = DEV_SUMMARY6B.iloc[1], DEV_SUMMARY6B.iloc[2]
_W, _R = "WRONG instructions acted on % (lower=better)", "  ...accepted on the RIGHT bowl %"
USE_CONTRAST = bool(len(_dev2["hybrid + contrast"]) == len(DEV6_STATES) * NUM_TASKS * len(KINDS)
                    and _c[_W] <= _h[_W] and _c[_R] >= _h[_R] and (_c[_W] < _h[_W] or _c[_R] > _h[_R]))
HYB_SCENE_VOCAB = list(CONTRAST_VOCAB) if USE_CONTRAST else []
DEV_CHOSEN6 = _c if USE_CONTRAST else _h
with open(DEV_CHOICE_JSON, "w") as f:
    json.dump({"chosen": "hybrid + contrast" if USE_CONTRAST else "hybrid", "scene_vocab": HYB_SCENE_VOCAB,
               "rule": "contrast kept only if wrong-acted <= plain AND accepted-on-right-bowl >= plain, with one strictly better",
               "decided_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f, indent=1)
print("\nSTEP 1b VERDICT:", "KEEP the contrast check (it will be frozen and used on the fresh test)" if USE_CONTRAST
      else "DROP the contrast check - the plain hybrid goes to the freeze")
print("Next: run the freeze cell.")


=== hybrid: 193 scenes to run (17 already saved) -> verify_owl_hybrid_contrast.csv
[1/193] t0 s2 relation_violation G=1.00 S_rel=0.00 D_rel=0.19 sel=other_bowl (~41.9 min left)
[2/193] t0 s2 distractor         G=0.88 S_rel=1.00 D_rel=1.00 sel=target     (~40.0 min left)
... (245 more lines of progress) ...
wrong instructions: plain acts where contrast stops in 4 scenes, contrast acts where plain stops in 0 (exact McNemar p = 0.125)
right instructions: plain accepts where contrast refuses in 15 scenes, contrast accepts where plain refuses in 0 (exact McNemar p = 0.000)

STEP 1b VERDICT: DROP the contrast check - the plain hybrid goes to the freeze


## Step 2: FREEZE the hybrid settings (*before any fresh scene*)

In [ ]:
# %% CELL v6_C_freeze — STEP 2: freeze the hybrid settings BEFORE any fresh test scene is touched (seconds)
assert "STEP1_OK" in globals(), "Run the STEP 1 cell first."
if not STEP1_OK:
    print("WARNING: step 1 did not say GO. Freezing anyway only makes sense if you decided to continue.")
_v4 = json.load(open(FROZEN_JSON))
_n = DEV_CHOSEN6 if "DEV_CHOSEN6" in globals() else DEV_SUMMARY6.iloc[1]
print("freezing the hybrid gate with contrast vocabulary:", HYB_SCENE_VOCAB if HYB_SCENE_VOCAB else "[] (contrast check off)")
FROZEN6 = {"name": "v6 hybrid gate (OWLv2 + Grounding DINO)",
           "detector": {"owlv2": "google/owlv2-base-patch16-ensemble", "grounding_dino": GDINO_MODEL_ID,
                        "grounding_dino_phrases": sorted(GD_PHRASES), "grounding_dino_fallback_if_owl_empty": sorted(GD_FALLBACK),
                        "owl_ramp": list(OWL_RAMP), "gdino_confidence_rescaled_from": [0.22, 0.40], "scene_vocab": list(HYB_SCENE_VOCAB)},
           "gate": {k: _v4[k] for k in ("weights", "high", "low", "reground", "mid_rule", "mid_min_rel", "mid_min_unamb")},
           "gate_source": "weights and thresholds unchanged from the first paper's frozen_config.json",
           "selected_on": f"development start states {DEV6_STATES} (detector choice only)",
           "dev_metrics": {k: (None if _n[k] != _n[k] else _n[k]) for k in ("right bowl % (valid)", "RIGHT instructions accepted %",
                                                                             "  ...accepted on the RIGHT bowl %",
                                                                             "WRONG instructions acted on % (lower=better)", "AUROC", "J")},
           "fresh_test_states": FRESH_STATES, "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S")}
if os.path.exists(FROZEN6_JSON):
    print("frozen_config_v6.json ALREADY EXISTS - it is NOT overwritten (delete it by hand only if you really mean to re-freeze).")
else:
    assert not os.path.exists(FRESH_GD_CSV) and not os.path.exists(FRESH_HY_CSV), "Fresh test files already exist - the freeze must come first."
    with open(FROZEN6_JSON, "w") as f:
        json.dump(FROZEN6, f, indent=1, default=float)
    print("FROZEN ->", FROZEN6_JSON)
_f = json.load(open(FROZEN6_JSON))
FROZEN6_HASH = hashlib.sha1(json.dumps(_f, sort_keys=True, default=float).encode()).hexdigest()[:10]
print(json.dumps(_f, indent=1))
print("\nhash:", FROZEN6_HASH, "| frozen at:", _f["frozen_at"], " <- write these two down for the paper")

freezing the hybrid gate with contrast vocabulary: [] (contrast check off)
FROZEN -> results/v6/frozen_config_v6.json
{
 "name": "v6 hybrid gate (OWLv2 + Grounding DINO)",
 "detector": {
  "owlv2": "google/owlv2-base-patch16-ensemble",
  "grounding_dino": "IDEA-Research/grounding-dino-base",
  "grounding_dino_phrases": [
   "box",
   "burner",
   "cabinet",
   "carton",
   "cookie box",
   "cooktop",
   "drawer",
   "stove",
   "wooden cabinet"
  ],
  "grounding_dino_fallback_if_owl_empty": [
   "dish",
   "plate",
   "white plate"
  ],
  "owl_ramp": [
   0.1,
   0.25
  ],
  "gdino_confidence_rescaled_from": [
   0.22,
   0.4
  ],
  "scene_vocab": []
 },
 "gate": {
  "weights": [
   0.266667,
   0.4,
   0.333333,
   0.0
  ],
  "high": 0.7,
  "low": 0.5,
  "reground": true,
  "mid_rule": true,
  "mid_min_rel": 0.3,
  "mid_min_unamb": 0.75
 },
 "gate_source": "weights and thresholds unchanged from the first paper's frozen_config.json",
 "selected_on": "development start states [0, 1, 2] 

## Step 3A: both gates on the 350 fresh scenes (about 1 h, resumable, no robot motion)

In [ ]:
# %% CELL v6_D_fresh_gate — STEP 3A: old gate and hybrid gate on the 350 FRESH scenes (~40-60 min, resumable, no robot)
assert os.path.exists(FROZEN6_JSON), "Run the freeze cell first: the settings must be frozen before any fresh scene is touched."
_f = json.load(open(FROZEN6_JSON))
assert (list(_f["detector"]["owl_ramp"]) == list(OWL_RAMP) and sorted(GD_PHRASES) == _f["detector"]["grounding_dino_phrases"]
        and list(_f["detector"]["scene_vocab"]) == list(HYB_SCENE_VOCAB)), \
    "The settings in memory differ from the frozen file - re-run the v6 setup cell without edits."
run_gate_sweep(FRESH_STATES, "gdino", FRESH_GD_CSV)
run_gate_sweep(FRESH_STATES, "hybrid", FRESH_HY_CSV)
_fr = {"GDINO (old)": decide_all(load6(FRESH_GD_CSV, FRESH_STATES), gate_cfg6("gdino")),
       "hybrid (new)": decide_all(load6(FRESH_HY_CSV, FRESH_STATES), gate_cfg6("hybrid"))}
compare_gates6(_fr, None, f"STEP 3A - GATE DECISIONS on FRESH states {FRESH_STATES} (task success comes after the rollouts)")
print("\nSTEP 3A DONE -> now run the rollout cell (overnight)")


=== gdino: 350 scenes to run (0 already saved) -> fresh_verify_gdino.csv
[1/350] t0 s5 clean              G=0.87 S_rel=0.33 D_rel=0.64 sel=other_bowl (~90.1 min left)
[2/350] t0 s5 absent_reference   G=0.00 S_rel=0.00 D_rel=0.00 sel=other_bowl (~82.4 min left)
... (751 more lines of progress) ...
8               55.0          80.0
9               75.0          75.0

STEP 3A DONE -> now run the rollout cell (overnight)


## Step 3B: SmolVLA rollouts of the 200 valid fresh scenes (about 5-6 h, resumable)
One rollout per scene serves both gates, because the gate never changes SmolVLA's actions.

In [ ]:
# %% CELL v6_E_fresh_rollouts — STEP 3B: SmolVLA rollouts on the fresh scenes (~5-6 h for the 200 valid ones; RESUMABLE)
# The gate never changes SmolVLA's actions, so one rollout per scene serves both gates.
ROLL_INVALID6 = False          # True = also roll out the 150 wrong-instruction scenes (about 4 more hours; optional)
_kinds = VALID_KINDS + (INVALID_KINDS_V4 if ROLL_INVALID6 else [])
_done = _done_keys(ROLL6_CSV, ["task_id", "state", "condition"])
_queue = [(t, s, k) for k in _kinds for s in FRESH_STATES for t in range(NUM_TASKS) if (str(t), str(s), k) not in _done]
print(f"rollouts to run: {len(_queue)} ({len(_done)} already saved)")
_t0 = time.time()
for i, (t, s, k) in enumerate(_queue):
    env, _, clean_instr = get_task_env_and_instruction(t)
    instr = perturb_instruction(clean_instr, k, s) if k in ("absent_reference", "absent_target") else clean_instr
    row = run_baseline_v4(env, t, s, instr, perturb=k if k in SCENE_PERTURBATIONS else None)
    env.close()
    OBS_HOOK = None
    row["condition"], row["split"] = k, "fresh"
    _append(ROLL6_CSV, ROLL_FIELDS, row)
    eta = (time.time() - _t0) / (i + 1) * (len(_queue) - i - 1) / 60
    print(f"[{i+1}/{len(_queue)}] t{t} s{s} {k:18s} SmolVLA success={row['success']} {row['failure_reason']} (~{eta:.0f} min left)")
print("STEP 3B ROLLOUTS DONE -> run the results cell")

rollouts to run: 83 (117 already saved)
[1/83] t7 s6 position_change    SmolVLA success=0 grasped_but_not_placed (~374 min left)
[2/83] t8 s6 position_change    SmolVLA success=0 grasped_but_not_placed (~363 min left)
[3/83] t9 s6 position_change    SmolVLA success=0 never_reached_target (~357 min left)
... (77 more lines of progress) ...
[81/83] t7 s9 occlusion          SmolVLA success=1 success (~5 min left)
[82/83] t8 s9 occlusion          SmolVLA success=1 success (~3 min left)
[83/83] t9 s9 occlusion          SmolVLA success=1 success (~0 min left)
STEP 3B ROLLOUTS DONE -> run the results cell


## Step 3C: final results on the fresh scenes (Table V: first gate and hybrid gate)

In [ ]:
# %% CELL v6_F_results — STEP 3C: FINAL RESULTS on the fresh test scenes (seconds).
_f = json.load(open(FROZEN6_JSON))
print("frozen config:", _f["name"], "| frozen at", _f["frozen_at"], "| hash",
      hashlib.sha1(json.dumps(_f, sort_keys=True, default=float).encode()).hexdigest()[:10])
roll6 = pd.read_csv(ROLL6_CSV).drop_duplicates(["task_id", "state", "condition"], keep="last")
rsm6 = {(int(r.task_id), int(r.state), r.condition): int(r.success) for r in roll6.itertuples()}
bsm6 = {(t, s): v for (t, s, k), v in rsm6.items() if k == "clean"}
GATES6 = {"GDINO (old)": decide_all(load6(FRESH_GD_CSV, FRESH_STATES), gate_cfg6("gdino")),
          "hybrid (new)": decide_all(load6(FRESH_HY_CSV, FRESH_STATES), gate_cfg6("hybrid"))}
for d in GATES6.values():
    d["smolvla_success"] = [rsm6.get((int(r.task_id), int(r.state), r.kind)) for r in d.itertuples()]
_n_exp = len(FRESH_STATES) * NUM_TASKS * len(KINDS)
print(f"fresh scenes per gate: {[len(d) for d in GATES6.values()]} (expected {_n_exp}) | rollouts found: {len(roll6)}")

FINAL_SUMMARY6 = compare_gates6(GATES6, bsm6, f"TABLE 1 - GATE COMPARISON on FRESH states {FRESH_STATES}")
FINAL_SUMMARY6.to_csv(os.path.join(V6_DIR, "final_summary.csv"), index=False)

# ---------- main table: per condition, with task success ----------
rows = []
for kind in KINDS:
    row = {"condition": kind, "N": int((GATES6["GDINO (old)"].kind == kind).sum())}
    for name, d in GATES6.items():
        g = d[d.kind == kind]
        tag = "old" if name.startswith("GDINO") else "new"
        k = int(g.executed.sum())
        row[f"{tag} acted %"] = pct(k / len(g)) if len(g) else float("nan")
        row[f"{tag} acted 95% CI"] = str(wilson(k, len(g)))
        if kind in VALID_KINDS:
            known = g[g.smolvla_success.notna()]
            acc = known[known.executed == 1]
            row["SmolVLA success %"] = pct(known.smolvla_success.mean()) if len(known) else float("nan")
            row[f"{tag} gated success %"] = pct((known.executed * known.smolvla_success).mean()) if len(known) else float("nan")
            row[f"{tag} success when acting %"] = pct(acc.smolvla_success.mean()) if len(acc) else float("nan")
            row["rolled out"] = f"{len(known)}/{len(g)}"
    rows.append(row)
main6 = pd.DataFrame(rows)
print("\nTABLE 2 - per condition (SmolVLA alone acts on 100% of every row)")
print(main6.to_string(index=False))
main6.to_csv(os.path.join(V6_DIR, "final_main_table.csv"), index=False)

# ---------- paired tests between the two gates, same scenes ----------
old, new = GATES6["GDINO (old)"], GATES6["hybrid (new)"]
a, b, p = paired6(old, new, 0)
print(f"\nwrong instructions ({int((old.valid == 0).sum())}): old acts / new stops in {a}; new acts / old stops in {b}; exact McNemar p = {p:.4f}")
a, b, p = paired6(old, new, 1)
print(f"right instructions ({int((old.valid == 1).sum())}): old accepts / new refuses in {a}; new accepts / old refuses in {b}; exact McNemar p = {p:.4f}")
for name, d in GATES6.items():
    inv = d[d.valid == 0]; k = int(inv.executed.sum())
    print(f"{name}: vs SmolVLA alone on wrong instructions: 100% -> {pct(k / len(inv))}% (95% CI {wilson(k, len(inv))}), "
          f"exact McNemar p = {mcnemar_exact(len(inv) - k, 0):.2e}")
    cl = d[(d.kind == "clean") & d.smolvla_success.notna()]
    if len(cl):
        lost = int(((cl.smolvla_success == 1) & (cl.executed == 0)).sum())
        print(f"   nominal success: SmolVLA {pct(cl.smolvla_success.mean())}% -> gated {pct((cl.smolvla_success * cl.executed).mean())}% "
              f"(successes lost {lost}, exact McNemar p = {mcnemar_exact(lost, 0):.4f})")
    av = d[(d.valid == 1) & (d.executed == 1)]
    wrong = int(sum(getattr(r, f"sel_{r.scene_used}") != "target" for r in av.itertuples()))
    print(f"   accepted valid scenes verified on the WRONG bowl: {wrong}/{len(av)} | label-uncertain valid scenes: "
          f"{int(d.label_uncertain.sum())} (accepted {int(d[d.label_uncertain == 1].executed.sum())})")

# ---------- head-to-head: new gate vs old gate, and each vs SmolVLA (success > stop > wrong/failed execution) ----------
def _outcome6(valid, executed, s):
    if not executed:
        return "stop"
    if valid == 0:
        return "bad"
    return None if s is None or s != s else ("success" if s == 1 else "bad")
_R = {"success": 2, "stop": 1, "bad": 0}
m = old[["task_id", "state", "kind", "valid", "executed", "smolvla_success"]].merge(
    new[["task_id", "state", "kind", "executed"]], on=["task_id", "state", "kind"], suffixes=("_old", "_new"))
h = []
for r in m.itertuples():
    o = {"smolvla": _outcome6(r.valid, 1, r.smolvla_success), "old": _outcome6(r.valid, r.executed_old, r.smolvla_success),
         "new": _outcome6(r.valid, r.executed_new, r.smolvla_success)}
    if None in o.values():
        continue
    h.append({"kind": r.kind, **o})
h = pd.DataFrame(h)
if len(h):
    def _vs(x, y):
        bx = int((h[x].map(_R) > h[y].map(_R)).sum()); by = int((h[x].map(_R) < h[y].map(_R)).sum())
        return f"{x} better {bx}, same {len(h) - bx - by}, {y} better {by} (exact sign test p = {mcnemar_exact(bx, by):.2e})"
    print(f"\nTABLE 3 - head-to-head on {len(h)} scenes with a known outcome:")
    print("  new gate vs old gate :", _vs("new", "old"))
    print("  new gate vs SmolVLA  :", _vs("new", "smolvla"))
    print("  old gate vs SmolVLA  :", _vs("old", "smolvla"))
    dist = pd.DataFrame({c: h[c].value_counts(normalize=True).reindex(["success", "stop", "bad"]).fillna(0) * 100
                         for c in ("smolvla", "old", "new")}).round(1)
    print("\noutcome of every scene (%):"); print(dist.to_string())
    dist.to_csv(os.path.join(V6_DIR, "final_outcomes.csv"))
for name, d in GATES6.items():
    d.drop(columns=[c for c in d.columns if c.startswith("ref_confs")]).to_csv(
        os.path.join(V6_DIR, f"final_decisions_{'old' if name.startswith('GDINO') else 'new'}.csv"), index=False)
print("\nALL RESULTS SAVED IN", V6_DIR)

frozen config: v6 hybrid gate (OWLv2 + Grounding DINO) | frozen at 2026-10-02 12:17:57 | hash a5f5591cec
fresh scenes per gate: [350, 350] (expected 350) | rollouts found: 200

TABLE 1 - GATE COMPARISON on FRESH states [5, 6, 7, 8, 9]
gate                                          GDINO (old) hybrid (new)
scenes                                                350          350
right bowl % (valid)                                 35.0         67.5
right reference %                                    45.6         52.2
RIGHT instructions accepted %                        42.0         52.5
  ...accepted on the RIGHT bowl %                    24.5         43.0
WRONG instructions acted on % (lower=better)         12.0          5.3
  95% CI                                      (7.7, 18.2)  (2.7, 10.2)
nominal success %                                    32.0         44.0
AUROC                                               0.738        0.875
J                                                     0

## Step 4: oracle object identity on the fresh scenes (analysis only, about 30 min, resumable)

In [ ]:
# %% CELL v6_G_oracle — STEP 4: corrected oracle-identity study on the fresh scenes (~30 min, resumable, no robot)
# Same frozen hybrid gate, but every box that is NOT the named object is removed (checked against the simulator).
# Fix vs the first paper: the cabinet and the stove are now checked too (before, all their boxes were deleted).
assert os.path.exists(FROZEN6_JSON), "Run the freeze cell first."
run_gate_sweep(FRESH_STATES, "hybrid", FRESH_ORC_CSV, oracle=True)
_or = {"GDINO (old)": decide_all(load6(FRESH_GD_CSV, FRESH_STATES), gate_cfg6("gdino")),
       "hybrid (new)": decide_all(load6(FRESH_HY_CSV, FRESH_STATES), gate_cfg6("hybrid")),
       "hybrid + oracle identity": decide_all(load6(FRESH_ORC_CSV, FRESH_STATES), gate_cfg6("hybrid"))}
_bs = None
if os.path.exists(ROLL6_CSV):
    _r = pd.read_csv(ROLL6_CSV).drop_duplicates(["task_id", "state", "condition"], keep="last")
    _bs = {(int(r.task_id), int(r.state)): int(r.success) for r in _r[_r.condition == "clean"].itertuples()}
ORACLE_SUMMARY6 = compare_gates6(_or, _bs, "STEP 4 - ORACLE OBJECT IDENTITY (upper bound for grounding), fresh scenes")
ORACLE_SUMMARY6.to_csv(os.path.join(V6_DIR, "oracle_summary.csv"), index=False)
_o = _or["hybrid + oracle identity"]
print("\nwith oracle identity, target bowl found on valid scenes:",
      pct((_o[_o.kind.isin(VALID_KINDS)].sel_0 == "target").mean()), "%  (the rest: the detector never boxed the instructed bowl)")


=== hybrid + oracle: 350 scenes to run (0 already saved) -> fresh_verify_hybrid_oracle.csv
[1/350] t0 s5 clean              G=1.00 S_rel=1.00 D_rel=1.00 sel=target     (~55.2 min left)
[2/350] t0 s5 absent_reference   G=0.00 S_rel=0.00 D_rel=0.00 sel=target     (~50.2 min left)
... (399 more lines of progress) ...
8               55.0          80.0                     100.0
9               75.0          75.0                      75.0

with oracle identity, target bowl found on valid scenes: 96.5 %  (the rest: the detector never boxed the instructed bowl)
